# Classification Report Demo

## Setup Data

In [1]:
from pathlib import Path
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from mlreport import ComparisonReport, Report

Path("reports").mkdir(exist_ok=True)

X, y = load_iris(return_X_y=True)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y,
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
author = "Lucas Summers"
theme = "light"

## Model 1: Train/Test Split Report

In [2]:
split_model = Pipeline(
    [
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=1000)),
    ]
)

split_model.fit(X_train, y_train)
y_pred_train = split_model.predict(X_train)
y_pred_test = split_model.predict(X_test)

split_report = Report(
    split_model,
    title="Iris Classification Report - Train/Test Split",
    author=author,
    description="Logistic Regression evaluated on a standard train/test split.",
    theme=theme,
)

split_report.add_split("train", X_train, y_train, y_pred_train)
split_report.add_split("test", X_test, y_test, y_pred_test)
split_report.build().to_pdf("reports/iris_split_report.pdf").to_txt()

Iris Classification Report - Train/Test Split
Lucas Summers
Logistic Regression evaluated on a standard train/test split.

Model Overview
+-----------------+--------------------+
| Property        | Value              |
+=================+====================+
| Name            | LogisticRegression |
+-----------------+--------------------+
| Type            | Classification     |
+-----------------+--------------------+
| Sklearn         | 1.8.0              |
+-----------------+--------------------+
| Parameter Count | 22                 |
+-----------------+--------------------+

Dataset Overview
+--------------------+-------------+
| Property           | Value       |
+====================+=============+
| Features           | 4           |
+--------------------+-------------+
| Total Observations | 150         |
+--------------------+-------------+
| CV Folds           | None        |
+--------------------+-------------+
| Train              | 112 (74.7%) |
+--------------------+-

## Model 2: CV Report

In [3]:
cv_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=4,
    random_state=42,
)
cv_model.fit(X_train, y_train)


cv_report = Report(
    cv_model,
    title="Iris Classification Report - Cross-Validation",
    author=author,
    description="Random Forest evaluated with 5-fold stratified cross-validation.",
    theme=theme,
)

cv_report.add_crossval(X_train, y_train, cv=cv)
cv_report.build().to_pdf("reports/iris_cv_report.pdf").to_txt()

Iris Classification Report - Cross-Validation
Lucas Summers
Random Forest evaluated with 5-fold stratified cross-validation.

Model Overview
+-----------------+------------------------+
| Property        | Value                  |
+=================+========================+
| Name            | RandomForestClassifier |
+-----------------+------------------------+
| Type            | Classification         |
+-----------------+------------------------+
| Sklearn         | 1.8.0                  |
+-----------------+------------------------+
| Parameter Count | 19                     |
+-----------------+------------------------+

Dataset Overview
+--------------------+---------+
| Property           |   Value |
+====================+=========+
| Features           |       4 |
+--------------------+---------+
| Total Observations |     112 |
+--------------------+---------+
| CV Folds           |       5 |
+--------------------+---------+

Class Distribution
+---------+------+-----------

## Model 3: Tuning + CV Report

In [4]:
search = GridSearchCV(
    estimator=Pipeline(
        [
            ("scaler", StandardScaler()),
            ("model", KNeighborsClassifier()),
        ]
    ),
    param_grid={
        "model__n_neighbors": [3, 5, 7, 9],
        "model__metric": ["euclidean", "manhattan"],
    },
    cv=cv,
    scoring="accuracy",
    n_jobs=-1,
)

search.fit(X_train, y_train)
best_tuned_model = search.best_estimator_

tuned_cv_report = Report(
    best_tuned_model,
    title="Iris Classification Report - Tuning + Cross-Validation",
    author=author,
    description="KNN tuned with GridSearchCV and evaluated with 5-fold CV.",
    theme=theme,
)

tuned_cv_report.add_crossval(X_train, y_train, cv=cv)
tuned_cv_report.add_search(search)
tuned_cv_report.build().to_pdf("reports/iris_tuned_cv_report.pdf").to_txt()

Iris Classification Report - Tuning + Cross-Validation
Lucas Summers
KNN tuned with GridSearchCV and evaluated with 5-fold CV.

Model Overview
+-----------------+----------------------+
| Property        | Value                |
+=================+======================+
| Name            | KNeighborsClassifier |
+-----------------+----------------------+
| Type            | Classification       |
+-----------------+----------------------+
| Sklearn         | 1.8.0                |
+-----------------+----------------------+
| Parameter Count | 16                   |
+-----------------+----------------------+

Dataset Overview
+--------------------+---------+
| Property           |   Value |
+====================+=========+
| Features           |       4 |
+--------------------+---------+
| Total Observations |     112 |
+--------------------+---------+
| CV Folds           |       5 |
+--------------------+---------+

Class Distribution
+---------+------+-------------+
|   Class |   Cv

## Comparison Report of 3 Models

In [5]:
comparison = ComparisonReport(
    reports=[
        split_report,
        cv_report,
        tuned_cv_report,
    ],
    title="Iris Workflow Comparison",
    author=author,
    description="Comparison of three classification workflows on the Iris dataset.",
    theme=theme,
)

comparison.build().to_pdf("reports/iris_comparison_report.pdf").to_txt()

Iris Workflow Comparison
Lucas Summers
Comparison of three classification workflows on the Iris dataset.

Models

Model 1
+-------------+---------------------------------------------------------------+
| Property    | Value                                                         |
+=============+===============================================================+
| Name        | LogisticRegression                                            |
+-------------+---------------------------------------------------------------+
| Description | Logistic Regression evaluated on a standard train/test split. |
+-------------+---------------------------------------------------------------+
| Type        | Classification                                                |
+-------------+---------------------------------------------------------------+
| Data        | Train/Test Split                                              |
+-------------+---------------------------------------------------------------